In [ ]:
!pip install -q torch transformers peft accelerate bitsandbytes "flwr[simulation]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/

In [ ]:
import importlib.metadata as metadata

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("CUDA available: torch not installed")

for package in ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "flwr"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

CUDA available: True
GPU: Tesla T4
torch: 2.11.0+cu128
transformers: 5.16.1
peft: 0.20.0
accelerate: 1.14.0
bitsandbytes: 0.50.2
flwr: 1.38.0


In [ ]:
!pip uninstall -y torch
!pip install torch --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp313-cp313-linux_x86_64.whl (780.4 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.26.0+cu128 requires torch==2.11.0, but you have torch 2.5.1+cu121 which is incompatible.


In [ ]:
import importlib.metadata as metadata
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("torch.__version__:", torch.__version__)

for package in ["torch", "torchvision", "transformers", "peft", "accelerate", "bitsandbytes", "flwr"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

CUDA available: True
GPU: Tesla T4
torch.__version__: 2.5.1+cu121
torch: 2.5.1+cu121
torchvision: 0.26.0+cu128
transformers: 5.16.1
peft: 0.20.0
accelerate: 1.14.0
bitsandbytes: 0.50.2
flwr: 1.38.0


In [ ]:
!find / -name "traffic_train.json" 2>/dev/null

In [ ]:
# Cell 1: Generate synthetic traffic data.
import os
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
steps_per_day = 24 * 6
n_steps = 14 * steps_per_day
timesteps = np.arange(n_steps)

daily_pattern = 100 + 35 * np.sin(2 * np.pi * (timesteps % steps_per_day) / steps_per_day - np.pi / 2)
day_index = timesteps // steps_per_day
weekend_multiplier = np.where((day_index % 7) >= 5, 0.7, 1.0)
traffic = daily_pattern * weekend_multiplier + rng.normal(0, 7, n_steps)

for _ in range(15):
    start = rng.integers(0, n_steps - 8)
    duration = rng.integers(2, 9)
    height = rng.uniform(30, 120)
    traffic[start:start + duration] += height * np.hanning(duration + 2)[1:-1]

traffic = np.clip(traffic, 0, None)
df = pd.DataFrame({"timestep": timesteps, "traffic": traffic.round(2)})

os.makedirs("data", exist_ok=True)
df.to_csv("data/synthetic_traffic.csv", index=False)

print(df.head(10))
print(f"\nMean: {df['traffic'].mean():.2f}")
print(f"Max: {df['traffic'].max():.2f}")

   timestep  traffic
0         0    67.13
1         1    57.75
2         2    70.39
3         3    71.88
4         4    51.87
5         5    56.71
6         6    67.09
7         7    64.41
8         8    66.99
9         9    61.69

Mean: 92.59
Max: 233.26


In [ ]:
# Cell 2: Create instruction-tuning forecasting examples.
import json
import numpy as np
import pandas as pd

df = pd.read_csv("data/synthetic_traffic.csv")
rng = np.random.default_rng(123)
history_length, future_length, n_examples = 12, 6, 200
max_start = len(df) - history_length - future_length

examples = []
for start in rng.choice(max_start, size=n_examples, replace=False):
    history = df["traffic"].iloc[start:start + history_length].tolist()
    future = df["traffic"].iloc[start + history_length:start + history_length + future_length].tolist()

    history_text = ", ".join(f"{v:.2f}" for v in history)
    prediction_text = ", ".join(f"{v:.2f}" for v in future)
    change = future[-1] - future[0]
    trend = "increasing" if change > 1 else "decreasing" if change < -1 else "stable"

    instruction = (
        "You are a traffic-forecasting assistant. Given the following 12 historical "
        "traffic values at 10-minute intervals, predict the next 6 values. Reason "
        "step by step about the recent trend, then end with exactly this format:\n"
        "PREDICTION: v1, v2, v3, v4, v5, v6\n\n"
        f"History: {history_text}"
    )
    output = f"Trend direction: {trend}.\nPREDICTION: {prediction_text}"
    examples.append({"instruction": instruction, "input": "", "output": output})

with open("data/traffic_train.json", "w", encoding="utf-8") as f:
    json.dump(examples, f, indent=2)

print(f"Saved {len(examples)} examples to data/traffic_train.json")

Saved 200 examples to data/traffic_train.json


In [ ]:
!ls -la data/ && python -c "import json; d = json.load(open('data/traffic_train.json')); print(f'Total examples: {len(d)}'); print(d[0])"

total 132
drwxr-xr-x 2 root root   4096 Sep 24 13:48 .
drwxr-xr-x 1 root root   4096 Sep 24 13:48 ..
-rw-r--r-- 1 root root  21627 Sep 24 13:48 synthetic_traffic.csv
-rw-r--r-- 1 root root 100882 Sep 24 13:48 traffic_train.json
Total examples: 200
{'instruction': 'You are a traffic-forecasting assistant. Given the following 12 historical traffic values at 10-minute intervals, predict the next 6 values. Reason step by step about the recent trend, then end with exactly this format:\nPREDICTION: v1, v2, v3, v4, v5, v6\n\nHistory: 83.02, 82.27, 71.40, 75.01, 87.43, 68.21, 60.49, 61.22, 73.40, 72.35, 70.25, 78.50', 'input': '', 'output': 'Trend direction: increasing.\nPREDICTION: 50.22, 70.64, 77.73, 57.85, 62.04, 58.39'}


In [ ]:
# Convert real synthetic traffic values into instruction-tuning examples.
import json
import numpy as np
import pandas as pd

df = pd.read_csv("data/synthetic_traffic.csv")
rng = np.random.default_rng(123)
history_length, future_length, n_examples = 12, 6, 200
max_start = len(df) - history_length - future_length

if max_start < n_examples:
    raise ValueError("Not enough data for 200 non-overlapping sampled windows.")

examples = []
for start in rng.choice(max_start + 1, size=n_examples, replace=False):
    history = df["traffic"].iloc[start:start + history_length].to_numpy()
    future = df["traffic"].iloc[start + history_length:start + history_length + future_length].to_numpy()

    first_3_mean = history[:3].mean()
    last_3_mean = history[-3:].mean()
    slope = last_3_mean - first_3_mean
    threshold = 2.0

    if slope > threshold:
        trend = "increasing"
    elif slope < -threshold:
        trend = "decreasing"
    else:
        trend = "stable"

    history_text = ", ".join(f"{value:.2f}" for value in history)
    prediction_text = ", ".join(f"{value:.2f}" for value in future)

    instruction = (
        "You are a traffic-forecasting assistant. Given 12 historical traffic values "
        "measured at 10-minute intervals, predict the next 6 values. Reason step by "
        "step about the trend, then end with exactly this line:\n"
        "PREDICTION: v1, v2, v3, v4, v5, v6\n\n"
        f"History: {history_text}"
    )
    output = (
        f"Trend direction: {trend}.\n"
        f"Justification: The average of the first 3 history values is {first_3_mean:.2f} "
        f"and the average of the last 3 is {last_3_mean:.2f}, giving a slope of {slope:.2f}.\n"
        f"PREDICTION: {prediction_text}"
    )
    examples.append({"instruction": instruction, "input": "", "output": output})

with open("data/traffic_train.json", "w", encoding="utf-8") as f:
    json.dump(examples, f, indent=2)

for i, example in enumerate(examples[:3], start=1):
    print(f"\n--- Example {i} ---")
    print(json.dumps(example, indent=2))


--- Example 1 ---
{
  "instruction": "You are a traffic-forecasting assistant. Given 12 historical traffic values measured at 10-minute intervals, predict the next 6 values. Reason step by step about the trend, then end with exactly this line:\nPREDICTION: v1, v2, v3, v4, v5, v6\n\nHistory: 83.02, 82.27, 71.40, 75.01, 87.43, 68.21, 60.49, 61.22, 73.40, 72.35, 70.25, 78.50",
  "input": "",
  "output": "Trend direction: decreasing.\nJustification: The average of the first 3 history values is 78.90 and the average of the last 3 is 73.70, giving a slope of -5.20.\nPREDICTION: 50.22, 70.64, 77.73, 57.85, 62.04, 58.39"
}

--- Example 2 ---
{
  "instruction": "You are a traffic-forecasting assistant. Given 12 historical traffic values measured at 10-minute intervals, predict the next 6 values. Reason step by step about the trend, then end with exactly this line:\nPREDICTION: v1, v2, v3, v4, v5, v6\n\nHistory: 136.59, 128.82, 133.93, 133.93, 137.65, 141.87, 127.29, 133.59, 144.54, 128.60, 127

In [ ]:
!python -c "import json; d = json.load(open('data/traffic_train.json')); print(f'Total examples: {len(d)}'); print(d[0]); print(); print(d[1])"

Total examples: 200
{'instruction': 'You are a traffic-forecasting assistant. Given 12 historical traffic values measured at 10-minute intervals, predict the next 6 values. Reason step by step about the trend, then end with exactly this line:\nPREDICTION: v1, v2, v3, v4, v5, v6\n\nHistory: 83.02, 82.27, 71.40, 75.01, 87.43, 68.21, 60.49, 61.22, 73.40, 72.35, 70.25, 78.50', 'input': '', 'output': 'Trend direction: decreasing.\nJustification: The average of the first 3 history values is 78.90 and the average of the last 3 is 73.70, giving a slope of -5.20.\nPREDICTION: 50.22, 70.64, 77.73, 57.85, 62.04, 58.39'}

{'instruction': 'You are a traffic-forecasting assistant. Given 12 historical traffic values measured at 10-minute intervals, predict the next 6 values. Reason step by step about the trend, then end with exactly this line:\nPREDICTION: v1, v2, v3, v4, v5, v6\n\nHistory: 136.59, 128.82, 133.93, 133.93, 137.65, 141.87, 127.29, 133.59, 144.54, 128.60, 127.62, 134.31', 'input': '', '

In [ ]:
# Create and verify instruction-tuning examples from real dataframe values.
import json
import os
import re
import numpy as np
import pandas as pd

df = pd.read_csv("data/synthetic_traffic.csv", usecols=["timestep", "traffic"])
rng = np.random.default_rng(123)
n_examples = 200
valid_starts = np.arange(len(df) - 18 + 1)

if len(valid_starts) < n_examples:
    raise ValueError("Not enough dataframe rows to create 200 valid windows.")

examples = []
expected_predictions = []

for i in rng.choice(valid_starts, size=n_examples, replace=False):
    history_values = df["traffic"].iloc[i : i + 12].tolist()
    future_values = df["traffic"].iloc[i + 12 : i + 18].tolist()

    slope = float(np.mean(history_values[-3:]) - np.mean(history_values[:3]))
    if slope > 1:
        trend = "increasing"
    elif slope < -1:
        trend = "decreasing"
    else:
        trend = "stable"

    history_text = ", ".join(f"{value:.2f}" for value in history_values)
    prediction_text = ", ".join(f"{value:.2f}" for value in future_values)

    instruction = (
        "You are a traffic-forecasting assistant. Predict the next 6 traffic values "
        "given the following 12 historical values measured at 10-minute intervals.\n\n"
        f"History: {history_text}\n\n"
        "Reason step by step about the traffic pattern, then end with exactly this format:\n"
        "PREDICTION: v1, v2, v3, v4, v5, v6"
    )
    output = (
        f"Trend direction: {trend}.\n"
        f"Justification: The history slope (mean of last 3 minus mean of first 3) is {slope:.2f}.\n"
        f"PREDICTION: {prediction_text}"
    )

    examples.append({"instruction": instruction, "input": "", "output": output})
    expected_predictions.append([float(f"{value:.2f}") for value in future_values])

for index, (example, expected) in enumerate(zip(examples, expected_predictions)):
    match = re.search(r"(?m)^PREDICTION:\s*(.+)$", example["output"])
    if not match:
        raise AssertionError(f"Example {index} failed: PREDICTION line not found.")

    try:
        parsed = [float(value.strip()) for value in match.group(1).split(",")]
    except ValueError as error:
        raise AssertionError(f"Example {index} failed: invalid PREDICTION values.") from error

    if len(parsed) != 6 or parsed != expected:
        raise AssertionError(
            f"Example {index} failed: parsed {parsed} does not match expected {expected}."
        )

os.makedirs("data", exist_ok=True)
with open("data/traffic_train.json", "w", encoding="utf-8") as f:
    json.dump(examples, f, indent=2)

print("All 200 examples verified: PREDICTION values match real dataframe data.")
for index, example in enumerate(examples[:2], start=1):
    print(f"\n--- Example {index} ---")
    print(json.dumps(example, indent=2))

All 200 examples verified: PREDICTION values match real dataframe data.

--- Example 1 ---
{
  "instruction": "You are a traffic-forecasting assistant. Predict the next 6 traffic values given the following 12 historical values measured at 10-minute intervals.\n\nHistory: 83.02, 82.27, 71.40, 75.01, 87.43, 68.21, 60.49, 61.22, 73.40, 72.35, 70.25, 78.50\n\nReason step by step about the traffic pattern, then end with exactly this format:\nPREDICTION: v1, v2, v3, v4, v5, v6",
  "input": "",
  "output": "Trend direction: decreasing.\nJustification: The history slope (mean of last 3 minus mean of first 3) is -5.20.\nPREDICTION: 50.22, 70.64, 77.73, 57.85, 62.04, 58.39"
}

--- Example 2 ---
{
  "instruction": "You are a traffic-forecasting assistant. Predict the next 6 traffic values given the following 12 historical values measured at 10-minute intervals.\n\nHistory: 136.59, 128.82, 133.93, 133.93, 137.65, 141.87, 127.29, 133.59, 144.54, 128.60, 127.62, 134.31\n\nReason step by step about t

In [ ]:
!python -c "import json; d = json.load(open('data/traffic_train.json')); print(f'Total examples: {len(d)}'); print(d[0]); print(); print(d[1])"

Total examples: 200
{'instruction': 'You are a traffic-forecasting assistant. Predict the next 6 traffic values given the following 12 historical values measured at 10-minute intervals.\n\nHistory: 83.02, 82.27, 71.40, 75.01, 87.43, 68.21, 60.49, 61.22, 73.40, 72.35, 70.25, 78.50\n\nReason step by step about the traffic pattern, then end with exactly this format:\nPREDICTION: v1, v2, v3, v4, v5, v6', 'input': '', 'output': 'Trend direction: decreasing.\nJustification: The history slope (mean of last 3 minus mean of first 3) is -5.20.\nPREDICTION: 50.22, 70.64, 77.73, 57.85, 62.04, 58.39'}

{'instruction': 'You are a traffic-forecasting assistant. Predict the next 6 traffic values given the following 12 historical values measured at 10-minute intervals.\n\nHistory: 136.59, 128.82, 133.93, 133.93, 137.65, 141.87, 127.29, 133.59, 144.54, 128.60, 127.62, 134.31\n\nReason step by step about the traffic pattern, then end with exactly this format:\nPREDICTION: v1, v2, v3, v4, v5, v6', 'input'

In [ ]:
!python3 -c "
import json, pandas as pd
d = json.load(open('data/traffic_train.json'))
df = pd.read_csv('data/synthetic_traffic.csv')
print('Total examples:', len(d))
print()
print('First example output:')
print(d[0]['output'])
print()
print('Does example 0 prediction appear anywhere in the real traffic column?')
import re
pred_line = [l for l in d[0]['output'].split(chr(10)) if 'PREDICTION' in l][0]
nums = [float(x) for x in pred_line.replace('PREDICTION:','').split(',')]
print('Predicted values:', nums)
print('First predicted value present in dataframe?', nums[0] in df['traffic'].values)
"

SyntaxError: unterminated string literal (detected at line 16) (2958270575.py, line 16)

In [ ]:
%%writefile check.py
import json
import pandas as pd

with open("data/traffic_train.json", "r", encoding="utf-8") as f:
    examples = json.load(f)

df = pd.read_csv("data/synthetic_traffic.csv")

print("Total examples:", len(examples))

first_output = examples[0]["output"]
prediction_line = next(line for line in first_output.splitlines() if "PREDICTION" in line)
parsed_numbers = [float(value.strip()) for value in prediction_line.split("PREDICTION:", 1)[1].split(",")]

print("Parsed prediction numbers:", parsed_numbers)

first_value_exists = any(
    round(float(traffic), 2) == round(parsed_numbers[0], 2)
    for traffic in df["traffic"]
)
print(first_value_exists)

print(first_output)

Writing check.py


In [ ]:
!python check.py

Total examples: 200
Parsed prediction numbers: [50.22, 70.64, 77.73, 57.85, 62.04, 58.39]
True
Trend direction: decreasing.
Justification: The history slope (mean of last 3 minus mean of first 3) is -5.20.
PREDICTION: 50.22, 70.64, 77.73, 57.85, 62.04, 58.39


In [ ]:
!nvidia-smi
!ls -la
!ls -la data/

Mon Sep 14 07:52:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install torch transformers peft accelerate bitsandbytes flwr[simulation] -q
!pip uninstall -y torch -q
!pip install torch --index-url https://download.pytorch.org/whl/cu121 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/

In [ ]:
!python3 -c "import torch; print('CUDA:', torch.cuda.is_available()); print('torch:', torch.__version__)"

CUDA: True
torch: 2.5.1+cu121


In [ ]:
import os
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
steps_per_day = 24 * 6
n_steps = 14 * steps_per_day
timesteps = np.arange(n_steps)

daily_pattern = 100 + 35 * np.sin(
    2 * np.pi * (timesteps % steps_per_day) / steps_per_day - np.pi / 2
)
day_index = timesteps // steps_per_day
weekend_multiplier = np.where((day_index % 7) >= 5, 0.7, 1.0)

traffic = daily_pattern * weekend_multiplier + rng.normal(0, 7, n_steps)

for _ in range(15):
    start = rng.integers(0, n_steps - 8)
    duration = rng.integers(2, 9)
    height = rng.uniform(30, 120)
    traffic[start:start + duration] += height * np.hanning(duration + 2)[1:-1]

df = pd.DataFrame({
    "timestep": timesteps,
    "traffic": np.clip(traffic, 0, None).round(2),
})

os.makedirs("data", exist_ok=True)
df.to_csv("data/synthetic_traffic.csv", index=False)

print(df.head(10))
print(f"\nMean: {df['traffic'].mean():.2f}")
print(f"Max: {df['traffic'].max():.2f}")

   timestep  traffic
0         0    67.13
1         1    57.75
2         2    70.39
3         3    71.88
4         4    51.87
5         5    56.71
6         6    67.09
7         7    64.41
8         8    66.99
9         9    61.69

Mean: 92.59
Max: 233.26


In [ ]:
!ls -la data/

total 32
drwxr-xr-x 2 root root  4096 Sep 14 08:36 .
drwxr-xr-x 1 root root  4096 Sep 14 08:36 ..
-rw-r--r-- 1 root root 21627 Sep 14 08:36 synthetic_traffic.csv


In [ ]:
# Create and verify instruction-tuning examples from real synthetic traffic data.
import json
import os
import re
import numpy as np
import pandas as pd

df = pd.read_csv("data/synthetic_traffic.csv", usecols=["timestep", "traffic"])
rng = np.random.default_rng(123)

n_examples = 200
valid_starts = np.arange(len(df) - 18 + 1)
if len(valid_starts) < n_examples:
    raise ValueError("Not enough rows to create 200 valid history/future windows.")

examples = []
expected_predictions = []

for i in rng.choice(valid_starts, size=n_examples, replace=False):
    history_values = df["traffic"].iloc[i : i + 12].tolist()
    future_values = df["traffic"].iloc[i + 12 : i + 18].tolist()

    slope = float(np.mean(history_values[-3:]) - np.mean(history_values[:3]))
    trend = "increasing" if slope > 1 else "decreasing" if slope < -1 else "stable"

    history_text = ", ".join(f"{value:.2f}" for value in history_values)
    prediction_text = ", ".join(f"{value:.2f}" for value in future_values)

    instruction = (
        "You are a traffic-forecasting assistant. Predict the next 6 traffic values "
        "given the following 12 historical traffic values measured at 10-minute intervals.\n\n"
        f"History: {history_text}\n\n"
        "Reason step by step about the traffic pattern, then end with exactly this format:\n"
        "PREDICTION: v1, v2, v3, v4, v5, v6"
    )
    output = (
        f"Trend direction: {trend}.\n"
        f"Justification: The history slope (mean of last 3 minus mean of first 3) is {slope:.2f}.\n"
        f"PREDICTION: {prediction_text}"
    )

    examples.append({"instruction": instruction, "input": "", "output": output})
    expected_predictions.append([float(f"{value:.2f}") for value in future_values])

for index, (example, expected) in enumerate(zip(examples, expected_predictions)):
    match = re.search(r"(?m)^PREDICTION:\s*(.+)$", example["output"])
    if not match:
        raise AssertionError(f"Example {index} failed: PREDICTION line not found.")

    parsed = [float(value.strip()) for value in match.group(1).split(",")]
    if len(parsed) != 6 or parsed != expected:
        raise AssertionError(
            f"Example {index} failed: parsed {parsed} does not match expected {expected}."
        )

os.makedirs("data", exist_ok=True)
with open("data/traffic_train.json", "w", encoding="utf-8") as f:
    json.dump(examples, f, indent=2)

print("All 200 examples verified: PREDICTION values match real dataframe data.")
for index, example in enumerate(examples[:2], start=1):
    print(f"\n--- Example {index} ---")
    print(json.dumps(example, indent=2))

All 200 examples verified: PREDICTION values match real dataframe data.

--- Example 1 ---
{
  "instruction": "You are a traffic-forecasting assistant. Predict the next 6 traffic values given the following 12 historical traffic values measured at 10-minute intervals.\n\nHistory: 83.02, 82.27, 71.40, 75.01, 87.43, 68.21, 60.49, 61.22, 73.40, 72.35, 70.25, 78.50\n\nReason step by step about the traffic pattern, then end with exactly this format:\nPREDICTION: v1, v2, v3, v4, v5, v6",
  "input": "",
  "output": "Trend direction: decreasing.\nJustification: The history slope (mean of last 3 minus mean of first 3) is -5.20.\nPREDICTION: 50.22, 70.64, 77.73, 57.85, 62.04, 58.39"
}

--- Example 2 ---
{
  "instruction": "You are a traffic-forecasting assistant. Predict the next 6 traffic values given the following 12 historical traffic values measured at 10-minute intervals.\n\nHistory: 136.59, 128.82, 133.93, 133.93, 137.65, 141.87, 127.29, 133.59, 144.54, 128.60, 127.62, 134.31\n\nReason step

In [ ]:
%%writefile check.py
import json
import pandas as pd

d = json.load(open('data/traffic_train.json'))
df = pd.read_csv('data/synthetic_traffic.csv')

print("Total examples:", len(d))
print()

output_text = d[0]['output']
pred_line = [line for line in output_text.split('\n') if 'PREDICTION' in line][0]
nums_str = pred_line.replace('PREDICTION:', '').strip()
nums = [float(x.strip()) for x in nums_str.split(',')]

print("Parsed prediction numbers:", nums)
print("First predicted value present in dataframe traffic column?",
      round(nums[0], 2) in df['traffic'].round(2).values)
print()
print("Full output text of example 0:")
print(output_text)

Overwriting check.py


In [ ]:
!python check.py

Total examples: 200

Parsed prediction numbers: [50.22, 70.64, 77.73, 57.85, 62.04, 58.39]
First predicted value present in dataframe traffic column? True

Full output text of example 0:
Trend direction: decreasing.
Justification: The history slope (mean of last 3 minus mean of first 3) is -5.20.
PREDICTION: 50.22, 70.64, 77.73, 57.85, 62.04, 58.39


In [ ]:
%%writefile client.py

Writing client.py


In [ ]:
%%writefile client.py

"""
FedFlow-LLM — client.py
------------------------
This defines what ONE simulated client (site) does:
  1. Load the shared base model + a small LoRA adapter on top of it.
  2. When the server sends the current adapter weights, load them in.
  3. Train locally, for a few steps, on this client's own slice of data.
  4. Send the updated adapter weights back to the server.

Nothing here is Flower-specific magic — it's a normal PyTorch/HuggingFace
training loop, wrapped in the shape Flower expects (a "NumPyClient").
"""
import os
os.environ["USE_TORCHVISION"] = "0"

import json
import sys

import flwr as fl
import numpy as np
import torch
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, get_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # swap to Qwen/Qwen2.5-1.5B-Instruct later (Day 4)
DATA_PATH = "data/traffic_train.json"
NUM_CLIENTS = 6
LOCAL_EPOCHS = 1          # how many passes over this client's own data slice, per round
LOCAL_BATCH_SIZE = 2
MAX_LENGTH = 256           # max tokens per training example (keeps this fast)


def load_model_and_tokenizer():
    """Load the base model once, and wrap it with a small LoRA adapter.
    Only the LoRA adapter's weights get trained/shared — the base model
    itself stays frozen, which is what keeps this small and fast."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cuda",
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],  # standard, small, fast LoRA target
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    return model, tokenizer


def load_client_data(client_id: int, num_clients: int):
    """Split the shared training JSON into num_clients roughly-equal
    slices, and return only this client's slice — simulating each
    client only ever seeing its own local data."""
    with open(DATA_PATH, "r") as f:
        all_examples = json.load(f)

    # Deterministic split: client_id 0 gets every 0th, num_clients-th example, etc.
    my_examples = all_examples[client_id::num_clients]
    return my_examples


def format_example(example, tokenizer):
    """Turn one {"instruction","input","output"} example into a single
    tokenized training string: instruction followed by the expected output."""
    full_text = example["instruction"] + "\n" + example["output"] + tokenizer.eos_token
    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    return tokens


class FedFlowClient(fl.client.NumPyClient):
    def __init__(self, client_id: int):
        self.client_id = client_id
        self.model, self.tokenizer = load_model_and_tokenizer()
        self.examples = load_client_data(client_id, NUM_CLIENTS)
        print(f"[Client {client_id}] loaded {len(self.examples)} local examples.")

    def get_parameters(self, config):
        """Send the current LoRA adapter weights to the server, as a list of numpy arrays."""
        state_dict = get_peft_model_state_dict(self.model)
        return [val.cpu().numpy() for val in state_dict.values()]

    def set_parameters(self, parameters):
        """Receive the server's current (averaged) LoRA weights and load them into this client's model."""
        state_dict = get_peft_model_state_dict(self.model)
        new_state_dict = {
            key: torch.tensor(val) for key, val in zip(state_dict.keys(), parameters)
        }
        set_peft_model_state_dict(self.model, new_state_dict)

    def fit(self, parameters, config):
        """This is the core federated step:
        1. Load the server's current weights.
        2. Train locally on this client's own data slice.
        3. Return the updated weights, plus how many examples were trained on
           (Flower uses this count to weight the averaging — FedAvg)."""
        self.set_parameters(parameters)
        self.model.train()

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4)

        total_loss = 0.0
        num_steps = 0

        for epoch in range(LOCAL_EPOCHS):
            for i in range(0, len(self.examples), LOCAL_BATCH_SIZE):
                batch = self.examples[i : i + LOCAL_BATCH_SIZE]
                if not batch:
                    continue

                for example in batch:
                    tokens = format_example(example, self.tokenizer)
                    input_ids = tokens["input_ids"].to("cuda")
                    attention_mask = tokens["attention_mask"].to("cuda")

                    outputs = self.model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=input_ids,
                    )
                    loss = outputs.loss

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    total_loss += loss.item()
                    num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        print(f"[Client {self.client_id}] local training done. avg_loss={avg_loss:.4f}")

        return self.get_parameters(config={}), len(self.examples), {"loss": avg_loss}

    def evaluate(self, parameters, config):
        """Minimal evaluation — reuses training loss as a rough signal.
        Good enough for a demo-scale project; not a held-out test set."""
        self.set_parameters(parameters)
        self.model.eval()

        total_loss = 0.0
        num_steps = 0
        with torch.no_grad():
            for example in self.examples[:10]:  # quick check on a small subset
                tokens = format_example(example, self.tokenizer)
                input_ids = tokens["input_ids"].to("cuda")
                attention_mask = tokens["attention_mask"].to("cuda")
                outputs = self.model(
                    input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
                )
                total_loss += outputs.loss.item()
                num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        return avg_loss, len(self.examples), {"loss": avg_loss}


def main():
    # client_id is passed as a command-line argument: python client.py 0
    client_id = int(sys.argv[1]) if len(sys.argv) > 1 else 0
    client = FedFlowClient(client_id)
    fl.client.start_client(
        server_address="127.0.0.1:8080",
        client=client.to_client(),
    )


if __name__ == "__main__":
    main()

Writing client.py


In [ ]:
%%writefile server.py

"""
FedFlow-LLM — server.py
------------------------
This runs the federated SIMULATION: it starts all 6 clients as
simulated processes on this one Colab GPU, coordinates NUM_ROUNDS
rounds of: send weights out -> clients train locally -> collect
updates back -> average them (FedAvg) -> repeat.

You don't call client.py directly — this file spins up all the
clients itself, in simulation mode, which is the standard way to
run federated learning demos on a single machine.
"""

import flwr as fl
from flwr.simulation import run_simulation
from flwr.client import ClientApp
from flwr.server import ServerApp, ServerConfig
from flwr.server.strategy import FedAvg

from client import FedFlowClient, NUM_CLIENTS

NUM_ROUNDS = 25  # Day 3 first test: keep this small (5). Bump to 20-25 for the real evidence run.


def client_fn(context):
    """Flower calls this once per simulated client to create it.
    context.node_config["partition-id"] tells us which client this is (0 to NUM_CLIENTS-1)."""
    client_id = int(context.node_config["partition-id"])
    return FedFlowClient(client_id).to_client()


def server_fn(context):
    """Defines the FedAvg strategy and how many rounds to run."""
    strategy = FedAvg(
        fraction_fit=1.0,          # use all 6 clients every round
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
    )
    config = ServerConfig(num_rounds=NUM_ROUNDS)
    return fl.server.ServerAppComponents(strategy=strategy, config=config)


client_app = ClientApp(client_fn=client_fn)
server_app = ServerApp(server_fn=server_fn)


if __name__ == "__main__":
    print(f"Starting FedFlow-LLM simulation: {NUM_CLIENTS} clients, {NUM_ROUNDS} rounds.")
    run_simulation(
        server_app=server_app,
        client_app=client_app,
        num_supernodes=NUM_CLIENTS,
        backend_config={"client_resources": {"num_cpus": 1, "num_gpus": 1.0 / NUM_CLIENTS}},
    )

Writing server.py


In [ ]:
!python3 -m py_compile client.py server.py && echo "Both files are syntactically valid."

Both files are syntactically valid.


In [ ]:
!python server.py

2026-09-24 13:51:21.875357: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/transformers/utils/import_utils.py", line 2531, in __getattr__
    module = self._get_module(self._class_to_module[name])
  File "/usr/local/lib/python3.13/dist-packages/transformers/utils/import_utils.py", line 2765, in _get_module
    raise e
  File "/usr/local/lib/python3.13/dist-packages/transformers/utils/import_utils.py", line 2763, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/importlib/__init__.py", line 88, in import_module
    return _bootstrap._gc

In [ ]:
!pip uninstall -y torchvision
!pip install torchvision --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.8 requires torchvision>=0.11, but you have torchvision 0.2.0 which is incompatible.


In [ ]:
!python3 -c "import torch, torchvision; print('torch:', torch.__version__); print('torchvision:', torchvision.__version__)"

torch: 2.5.1+cu121
torchvision: 0.2.0


In [ ]:
!python server.py

2026-09-24 13:52:23.246631: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/transformers/utils/import_utils.py", line 2531, in __getattr__
    module = self._get_module(self._class_to_module[name])
  File "/usr/local/lib/python3.13/dist-packages/transformers/utils/import_utils.py", line 2765, in _get_module
    raise e
  File "/usr/local/lib/python3.13/dist-packages/transformers/utils/import_utils.py", line 2763, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/importlib/__init__.py", line 88, in import_module
    return _bootstrap._gc

In [ ]:
!pip uninstall -y torchvision

import os

os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["USE_TORCHVISION"] = "0"

with open("run_config.py", "w", encoding="utf-8") as f:
    f.write(
        'import os\n'
        'os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"\n'
        'os.environ["USE_TORCHVISION"] = "0"\n'
    )

Found existing installation: torchvision 0.2.0
Uninstalling torchvision-0.2.0:
  Successfully uninstalled torchvision-0.2.0


In [ ]:
!pip uninstall -y torchvision -q

In [ ]:
%%writefile client.py

"""
FedFlow-LLM — client.py
------------------------
This defines what ONE simulated client (site) does:
  1. Load the shared base model + a small LoRA adapter on top of it.
  2. When the server sends the current adapter weights, load them in.
  3. Train locally, for a few steps, on this client's own slice of data.
  4. Send the updated adapter weights back to the server.

Nothing here is Flower-specific magic — it's a normal PyTorch/HuggingFace
training loop, wrapped in the shape Flower expects (a "NumPyClient").
"""

# Must be set BEFORE importing transformers/peft — tells transformers to
# never attempt to import torchvision internally. This project is
# text-only (traffic forecasting), so torchvision is never needed, and
# this permanently removes an entire class of torch/torchvision version
# mismatch errors rather than trying to keep both versions in sync.
import os
os.environ["USE_TORCHVISION"] = "0"

import json
import sys

import flwr as fl
import numpy as np
import torch
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, get_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # swap to Qwen/Qwen2.5-1.5B-Instruct later (Day 4)
DATA_PATH = "data/traffic_train.json"
NUM_CLIENTS = 6
LOCAL_EPOCHS = 1          # how many passes over this client's own data slice, per round
LOCAL_BATCH_SIZE = 2
MAX_LENGTH = 256           # max tokens per training example (keeps this fast)


def load_model_and_tokenizer():
    """Load the base model once, and wrap it with a small LoRA adapter.
    Only the LoRA adapter's weights get trained/shared — the base model
    itself stays frozen, which is what keeps this small and fast."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cuda",
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],  # standard, small, fast LoRA target
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    return model, tokenizer


def load_client_data(client_id: int, num_clients: int):
    """Split the shared training JSON into num_clients roughly-equal
    slices, and return only this client's slice — simulating each
    client only ever seeing its own local data."""
    with open(DATA_PATH, "r") as f:
        all_examples = json.load(f)

    # Deterministic split: client_id 0 gets every 0th, num_clients-th example, etc.
    my_examples = all_examples[client_id::num_clients]
    return my_examples


def format_example(example, tokenizer):
    """Turn one {"instruction","input","output"} example into a single
    tokenized training string: instruction followed by the expected output."""
    full_text = example["instruction"] + "\n" + example["output"] + tokenizer.eos_token
    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    return tokens


class FedFlowClient(fl.client.NumPyClient):
    def __init__(self, client_id: int):
        self.client_id = client_id
        self.model, self.tokenizer = load_model_and_tokenizer()
        self.examples = load_client_data(client_id, NUM_CLIENTS)
        print(f"[Client {client_id}] loaded {len(self.examples)} local examples.")

    def get_parameters(self, config):
        """Send the current LoRA adapter weights to the server, as a list of numpy arrays."""
        state_dict = get_peft_model_state_dict(self.model)
        return [val.cpu().numpy() for val in state_dict.values()]

    def set_parameters(self, parameters):
        """Receive the server's current (averaged) LoRA weights and load them into this client's model."""
        state_dict = get_peft_model_state_dict(self.model)
        new_state_dict = {
            key: torch.tensor(val) for key, val in zip(state_dict.keys(), parameters)
        }
        set_peft_model_state_dict(self.model, new_state_dict)

    def fit(self, parameters, config):
        """This is the core federated step:
        1. Load the server's current weights.
        2. Train locally on this client's own data slice.
        3. Return the updated weights, plus how many examples were trained on
           (Flower uses this count to weight the averaging — FedAvg)."""
        self.set_parameters(parameters)
        self.model.train()

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4)

        total_loss = 0.0
        num_steps = 0

        for epoch in range(LOCAL_EPOCHS):
            for i in range(0, len(self.examples), LOCAL_BATCH_SIZE):
                batch = self.examples[i : i + LOCAL_BATCH_SIZE]
                if not batch:
                    continue

                for example in batch:
                    tokens = format_example(example, self.tokenizer)
                    input_ids = tokens["input_ids"].to("cuda")
                    attention_mask = tokens["attention_mask"].to("cuda")

                    outputs = self.model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=input_ids,
                    )
                    loss = outputs.loss

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    total_loss += loss.item()
                    num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        print(f"[Client {self.client_id}] local training done. avg_loss={avg_loss:.4f}")

        return self.get_parameters(config={}), len(self.examples), {"loss": avg_loss}

    def evaluate(self, parameters, config):
        """Minimal evaluation — reuses training loss as a rough signal.
        Good enough for a demo-scale project; not a held-out test set."""
        self.set_parameters(parameters)
        self.model.eval()

        total_loss = 0.0
        num_steps = 0
        with torch.no_grad():
            for example in self.examples[:10]:  # quick check on a small subset
                tokens = format_example(example, self.tokenizer)
                input_ids = tokens["input_ids"].to("cuda")
                attention_mask = tokens["attention_mask"].to("cuda")
                outputs = self.model(
                    input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
                )
                total_loss += outputs.loss.item()
                num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        return avg_loss, len(self.examples), {"loss": avg_loss}


def main():
    # client_id is passed as a command-line argument: python client.py 0
    client_id = int(sys.argv[1]) if len(sys.argv) > 1 else 0
    client = FedFlowClient(client_id)
    fl.client.start_client(
        server_address="127.0.0.1:8080",
        client=client.to_client(),
    )


if __name__ == "__main__":
    main()

Overwriting client.py


In [ ]:
!python3 -m py_compile client.py server.py && echo "Both files are syntactically valid."

Both files are syntactically valid.


In [ ]:
!python server.py

2026-09-24 13:53:38.778601: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/content/server.py", line 21, in <module>
    from client import FedFlowClient, NUM_CLIENTS
  File "/content/client.py", line 29, in <module>
    from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, get_peft_model_state_dict
  File "/usr/local/lib/python3.13/dist-packages/peft/__init__.py", line 17, in <module>
    from .auto import (
    ...<8 lines>...
    )
  File "/usr/local/lib/python3.13/dist-packages/peft/auto.py", line 31, in <module>
    from .config import PeftConfig
  File "/usr/local/lib/python3.13/dist-packages/peft/config.py", line 30, in <module>
    from .utils import CONFIG_NAME, PeftType, Task

In [ ]:
!pip uninstall -y torchaudio -q

In [ ]:
%%writefile client.py

Overwriting client.py


In [ ]:
%%writefile client.py

"""
FedFlow-LLM — client.py
------------------------
This defines what ONE simulated client (site) does:
  1. Load the shared base model + a small LoRA adapter on top of it.
  2. When the server sends the current adapter weights, load them in.
  3. Train locally, for a few steps, on this client's own slice of data.
  4. Send the updated adapter weights back to the server.

Nothing here is Flower-specific magic — it's a normal PyTorch/HuggingFace
training loop, wrapped in the shape Flower expects (a "NumPyClient").
"""

# Must be set BEFORE importing transformers/peft — these environment
# variables tell transformers to skip its optional vision/audio code
# paths entirely, rather than trying to import torchvision/torchaudio.
# This project is text-only (traffic forecasting), so none of this is
# needed — and unlike patching torchvision/torchaudio one at a time,
# this blocks the whole multimodal import path in one shot.
import os
os.environ["USE_TORCHVISION"] = "0"
os.environ["USE_TORCHAUDIO"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import json
import sys

import flwr as fl
import numpy as np
import torch
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, get_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # swap to Qwen/Qwen2.5-1.5B-Instruct later (Day 4)
DATA_PATH = "data/traffic_train.json"
NUM_CLIENTS = 6
LOCAL_EPOCHS = 1          # how many passes over this client's own data slice, per round
LOCAL_BATCH_SIZE = 2
MAX_LENGTH = 256           # max tokens per training example (keeps this fast)


def load_model_and_tokenizer():
    """Load the base model once, and wrap it with a small LoRA adapter.
    Only the LoRA adapter's weights get trained/shared — the base model
    itself stays frozen, which is what keeps this small and fast."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cuda",
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],  # standard, small, fast LoRA target
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    return model, tokenizer


def load_client_data(client_id: int, num_clients: int):
    """Split the shared training JSON into num_clients roughly-equal
    slices, and return only this client's slice — simulating each
    client only ever seeing its own local data."""
    with open(DATA_PATH, "r") as f:
        all_examples = json.load(f)

    # Deterministic split: client_id 0 gets every 0th, num_clients-th example, etc.
    my_examples = all_examples[client_id::num_clients]
    return my_examples


def format_example(example, tokenizer):
    """Turn one {"instruction","input","output"} example into a single
    tokenized training string: instruction followed by the expected output."""
    full_text = example["instruction"] + "\n" + example["output"] + tokenizer.eos_token
    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    return tokens


class FedFlowClient(fl.client.NumPyClient):
    def __init__(self, client_id: int):
        self.client_id = client_id
        self.model, self.tokenizer = load_model_and_tokenizer()
        self.examples = load_client_data(client_id, NUM_CLIENTS)
        print(f"[Client {client_id}] loaded {len(self.examples)} local examples.")

    def get_parameters(self, config):
        """Send the current LoRA adapter weights to the server, as a list of numpy arrays."""
        state_dict = get_peft_model_state_dict(self.model)
        return [val.cpu().numpy() for val in state_dict.values()]

    def set_parameters(self, parameters):
        """Receive the server's current (averaged) LoRA weights and load them into this client's model."""
        state_dict = get_peft_model_state_dict(self.model)
        new_state_dict = {
            key: torch.tensor(val) for key, val in zip(state_dict.keys(), parameters)
        }
        set_peft_model_state_dict(self.model, new_state_dict)

    def fit(self, parameters, config):
        """This is the core federated step:
        1. Load the server's current weights.
        2. Train locally on this client's own data slice.
        3. Return the updated weights, plus how many examples were trained on
           (Flower uses this count to weight the averaging — FedAvg)."""
        self.set_parameters(parameters)
        self.model.train()

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4)

        total_loss = 0.0
        num_steps = 0

        for epoch in range(LOCAL_EPOCHS):
            for i in range(0, len(self.examples), LOCAL_BATCH_SIZE):
                batch = self.examples[i : i + LOCAL_BATCH_SIZE]
                if not batch:
                    continue

                for example in batch:
                    tokens = format_example(example, self.tokenizer)
                    input_ids = tokens["input_ids"].to("cuda")
                    attention_mask = tokens["attention_mask"].to("cuda")

                    outputs = self.model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=input_ids,
                    )
                    loss = outputs.loss

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    total_loss += loss.item()
                    num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        print(f"[Client {self.client_id}] local training done. avg_loss={avg_loss:.4f}")

        return self.get_parameters(config={}), len(self.examples), {"loss": avg_loss}

    def evaluate(self, parameters, config):
        """Minimal evaluation — reuses training loss as a rough signal.
        Good enough for a demo-scale project; not a held-out test set."""
        self.set_parameters(parameters)
        self.model.eval()

        total_loss = 0.0
        num_steps = 0
        with torch.no_grad():
            for example in self.examples[:10]:  # quick check on a small subset
                tokens = format_example(example, self.tokenizer)
                input_ids = tokens["input_ids"].to("cuda")
                attention_mask = tokens["attention_mask"].to("cuda")
                outputs = self.model(
                    input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
                )
                total_loss += outputs.loss.item()
                num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        return avg_loss, len(self.examples), {"loss": avg_loss}


def main():
    # client_id is passed as a command-line argument: python client.py 0
    client_id = int(sys.argv[1]) if len(sys.argv) > 1 else 0
    client = FedFlowClient(client_id)
    fl.client.start_client(
        server_address="127.0.0.1:8080",
        client=client.to_client(),
    )


if __name__ == "__main__":
    main()

Overwriting client.py


In [ ]:
!pip install "transformers==4.46.3" "peft==0.13.2" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 50.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.29 requires torchvision, which is not installed.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
%%writefile client.py

"""
FedFlow-LLM — client.py
------------------------
This defines what ONE simulated client (site) does:
  1. Load the shared base model + a small LoRA adapter on top of it.
  2. When the server sends the current adapter weights, load them in.
  3. Train locally, for a few steps, on this client's own slice of data.
  4. Send the updated adapter weights back to the server.

Nothing here is Flower-specific magic — it's a normal PyTorch/HuggingFace
training loop, wrapped in the shape Flower expects (a "NumPyClient").
"""

# Must be set BEFORE importing transformers/peft — these environment
# variables tell transformers to skip its optional vision/audio code
# paths entirely, rather than trying to import torchvision/torchaudio.
# This project is text-only (traffic forecasting), so none of this is
# needed — and unlike patching torchvision/torchaudio one at a time,
# this blocks the whole multimodal import path in one shot.
import os
os.environ["USE_TORCHVISION"] = "0"
os.environ["USE_TORCHAUDIO"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import json
import sys

import flwr as fl
import numpy as np
import torch
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, get_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # swap to Qwen/Qwen2.5-1.5B-Instruct later (Day 4)
DATA_PATH = "data/traffic_train.json"
NUM_CLIENTS = 6
LOCAL_EPOCHS = 1          # how many passes over this client's own data slice, per round
LOCAL_BATCH_SIZE = 2
MAX_LENGTH = 256           # max tokens per training example (keeps this fast)


def load_model_and_tokenizer():
    """Load the base model once, and wrap it with a small LoRA adapter.
    Only the LoRA adapter's weights get trained/shared — the base model
    itself stays frozen, which is what keeps this small and fast."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cuda",
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],  # standard, small, fast LoRA target
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    return model, tokenizer


def load_client_data(client_id: int, num_clients: int):
    """Split the shared training JSON into num_clients roughly-equal
    slices, and return only this client's slice — simulating each
    client only ever seeing its own local data."""
    with open(DATA_PATH, "r") as f:
        all_examples = json.load(f)

    # Deterministic split: client_id 0 gets every 0th, num_clients-th example, etc.
    my_examples = all_examples[client_id::num_clients]
    return my_examples


def format_example(example, tokenizer):
    """Turn one {"instruction","input","output"} example into a single
    tokenized training string: instruction followed by the expected output."""
    full_text = example["instruction"] + "\n" + example["output"] + tokenizer.eos_token
    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    return tokens


class FedFlowClient(fl.client.NumPyClient):
    def __init__(self, client_id: int):
        self.client_id = client_id
        self.model, self.tokenizer = load_model_and_tokenizer()
        self.examples = load_client_data(client_id, NUM_CLIENTS)
        print(f"[Client {client_id}] loaded {len(self.examples)} local examples.")

    def get_parameters(self, config):
        """Send the current LoRA adapter weights to the server, as a list of numpy arrays."""
        state_dict = get_peft_model_state_dict(self.model)
        return [val.cpu().numpy() for val in state_dict.values()]

    def set_parameters(self, parameters):
        """Receive the server's current (averaged) LoRA weights and load them into this client's model."""
        state_dict = get_peft_model_state_dict(self.model)
        new_state_dict = {
            key: torch.tensor(val) for key, val in zip(state_dict.keys(), parameters)
        }
        set_peft_model_state_dict(self.model, new_state_dict)

    def fit(self, parameters, config):
        """This is the core federated step:
        1. Load the server's current weights.
        2. Train locally on this client's own data slice.
        3. Return the updated weights, plus how many examples were trained on
           (Flower uses this count to weight the averaging — FedAvg)."""
        self.set_parameters(parameters)
        self.model.train()

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4)

        total_loss = 0.0
        num_steps = 0

        for epoch in range(LOCAL_EPOCHS):
            for i in range(0, len(self.examples), LOCAL_BATCH_SIZE):
                batch = self.examples[i : i + LOCAL_BATCH_SIZE]
                if not batch:
                    continue

                for example in batch:
                    tokens = format_example(example, self.tokenizer)
                    input_ids = tokens["input_ids"].to("cuda")
                    attention_mask = tokens["attention_mask"].to("cuda")

                    outputs = self.model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=input_ids,
                    )
                    loss = outputs.loss

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    total_loss += loss.item()
                    num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        print(f"[Client {self.client_id}] local training done. avg_loss={avg_loss:.4f}")

        return self.get_parameters(config={}), len(self.examples), {"loss": avg_loss}

    def evaluate(self, parameters, config):
        """Minimal evaluation — reuses training loss as a rough signal.
        Good enough for a demo-scale project; not a held-out test set."""
        self.set_parameters(parameters)
        self.model.eval()

        total_loss = 0.0
        num_steps = 0
        with torch.no_grad():
            for example in self.examples[:10]:  # quick check on a small subset
                tokens = format_example(example, self.tokenizer)
                input_ids = tokens["input_ids"].to("cuda")
                attention_mask = tokens["attention_mask"].to("cuda")
                outputs = self.model(
                    input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
                )
                total_loss += outputs.loss.item()
                num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        return avg_loss, len(self.examples), {"loss": avg_loss}


def main():
    # client_id is passed as a command-line argument: python client.py 0
    client_id = int(sys.argv[1]) if len(sys.argv) > 1 else 0
    client = FedFlowClient(client_id)
    fl.client.start_client(
        server_address="127.0.0.1:8080",
        client=client.to_client(),
    )


if __name__ == "__main__":
    main()

Overwriting client.py


In [ ]:
!python3 -m py_compile client.py server.py && echo "OK"

OK


In [ ]:
!python server.py

2026-09-14 09:02:02.462064: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Starting FedFlow-LLM simulation: 6 clients, 5 rounds.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
DEBUG:flwr:backend_config: {'client_resources': {'num_cpus': 1, 'num_gpus': 0.16666666666666666}}
DEBUG:flwr:Pre-registering run with id 14618551925650379678
DEBUG:flwr:Using InMemoryObjectStore
DEBUG:flwr:Using InMemoryLinkState
INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO:flwr:Starting Flower ServerApp, config: num_rounds=5, no round_timeou

In [ ]:
%%writefile server.py

"""
FedFlow-LLM — server.py
------------------------
This runs the federated SIMULATION: it starts all 6 clients as
simulated processes on this one Colab GPU, coordinates NUM_ROUNDS
rounds of: send weights out -> clients train locally -> collect
updates back -> average them (FedAvg) -> repeat.

You don't call client.py directly — this file spins up all the
clients itself, in simulation mode, which is the standard way to
run federated learning demos on a single machine.
"""

import flwr as fl
from flwr.simulation import run_simulation
from flwr.client import ClientApp
from flwr.server import ServerApp, ServerConfig
from flwr.server.strategy import FedAvg

from client import FedFlowClient, NUM_CLIENTS

NUM_ROUNDS = 25  # Day 3 first test: keep this small (5). Bump to 20-25 for the real evidence run.


def client_fn(context):
    """Flower calls this once per simulated client to create it.
    context.node_config["partition-id"] tells us which client this is (0 to NUM_CLIENTS-1)."""
    client_id = int(context.node_config["partition-id"])
    return FedFlowClient(client_id).to_client()


def server_fn(context):
    """Defines the FedAvg strategy and how many rounds to run."""
    strategy = FedAvg(
        fraction_fit=1.0,          # use all 6 clients every round
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
    )
    config = ServerConfig(num_rounds=NUM_ROUNDS)
    return fl.server.ServerAppComponents(strategy=strategy, config=config)


client_app = ClientApp(client_fn=client_fn)
server_app = ServerApp(server_fn=server_fn)


if __name__ == "__main__":
    print(f"Starting FedFlow-LLM simulation: {NUM_CLIENTS} clients, {NUM_ROUNDS} rounds.")
    run_simulation(
        server_app=server_app,
        client_app=client_app,
        num_supernodes=NUM_CLIENTS,
        backend_config={"client_resources": {"num_cpus": 1, "num_gpus": 1.0 / NUM_CLIENTS}},
    )

Overwriting server.py


In [ ]:
!python server.py

2026-09-24 13:55:28.135749: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Starting FedFlow-LLM simulation: 6 clients, 25 rounds.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
DEBUG:flwr:backend_config: {'client_resources': {'num_cpus': 1, 'num_gpus': 0.16666666666666666}}
DEBUG:flwr:Pre-registering run with id 7550795514687498299
DEBUG:flwr:Using InMemoryObjectStore
DEBUG:flwr:Using InMemoryLinkState
INFO :      Starting Flower ServerApp, config: num_rounds=25, no round_timeout
INFO:flwr:Starting Flower ServerApp, config: num_rounds=25, no round_time

In [ ]:
!nvidia-smi

Sun Sep 27 09:10:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
"""
FedFlow-LLM — client.py
------------------------
This defines what ONE simulated client (site) does:
  1. Load the shared base model + a small LoRA adapter on top of it.
  2. When the server sends the current adapter weights, load them in.
  3. Train locally, for a few steps, on this client's own slice of data.
  4. Send the updated adapter weights back to the server.

Nothing here is Flower-specific magic — it's a normal PyTorch/HuggingFace
training loop, wrapped in the shape Flower expects (a "NumPyClient").
"""

# Must be set BEFORE importing transformers/peft — these environment
# variables tell transformers to skip its optional vision/audio code
# paths entirely, rather than trying to import torchvision/torchaudio.
# This project is text-only (traffic forecasting), so none of this is
# needed — and unlike patching torchvision/torchaudio one at a time,
# this blocks the whole multimodal import path in one shot.
import os
os.environ["USE_TORCHVISION"] = "0"
os.environ["USE_TORCHAUDIO"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import json
import sys

import flwr as fl
import numpy as np
import torch
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, get_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # swapped from 0.5B — matches the Objectives slide's stated target
DATA_PATH = "data/traffic_train.json"
NUM_CLIENTS = 6
LOCAL_EPOCHS = 1          # how many passes over this client's own data slice, per round
LOCAL_BATCH_SIZE = 2
MAX_LENGTH = 256           # max tokens per training example (keeps this fast)


def load_model_and_tokenizer():
    """Load the base model once, and wrap it with a small LoRA adapter.
    Only the LoRA adapter's weights get trained/shared — the base model
    itself stays frozen, which is what keeps this small and fast."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cuda",
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],  # standard, small, fast LoRA target
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    return model, tokenizer


def load_client_data(client_id: int, num_clients: int):
    """Split the shared training JSON into num_clients roughly-equal
    slices, and return only this client's slice — simulating each
    client only ever seeing its own local data."""
    with open(DATA_PATH, "r") as f:
        all_examples = json.load(f)

    # Deterministic split: client_id 0 gets every 0th, num_clients-th example, etc.
    my_examples = all_examples[client_id::num_clients]
    return my_examples


def format_example(example, tokenizer):
    """Turn one {"instruction","input","output"} example into a single
    tokenized training string: instruction followed by the expected output."""
    full_text = example["instruction"] + "\n" + example["output"] + tokenizer.eos_token
    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    return tokens


class FedFlowClient(fl.client.NumPyClient):
    def __init__(self, client_id: int):
        self.client_id = client_id
        self.model, self.tokenizer = load_model_and_tokenizer()
        self.examples = load_client_data(client_id, NUM_CLIENTS)
        print(f"[Client {client_id}] loaded {len(self.examples)} local examples.")

    def get_parameters(self, config):
        """Send the current LoRA adapter weights to the server, as a list of numpy arrays."""
        state_dict = get_peft_model_state_dict(self.model)
        return [val.cpu().numpy() for val in state_dict.values()]

    def set_parameters(self, parameters):
        """Receive the server's current (averaged) LoRA weights and load them into this client's model."""
        state_dict = get_peft_model_state_dict(self.model)
        new_state_dict = {
            key: torch.tensor(val) for key, val in zip(state_dict.keys(), parameters)
        }
        set_peft_model_state_dict(self.model, new_state_dict)

    def fit(self, parameters, config):
        """This is the core federated step:
        1. Load the server's current weights.
        2. Train locally on this client's own data slice.
        3. Return the updated weights, plus how many examples were trained on
           (Flower uses this count to weight the averaging — FedAvg)."""
        self.set_parameters(parameters)
        self.model.train()

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4)

        total_loss = 0.0
        num_steps = 0

        for epoch in range(LOCAL_EPOCHS):
            for i in range(0, len(self.examples), LOCAL_BATCH_SIZE):
                batch = self.examples[i : i + LOCAL_BATCH_SIZE]
                if not batch:
                    continue

                for example in batch:
                    tokens = format_example(example, self.tokenizer)
                    input_ids = tokens["input_ids"].to("cuda")
                    attention_mask = tokens["attention_mask"].to("cuda")

                    outputs = self.model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=input_ids,
                    )
                    loss = outputs.loss

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    total_loss += loss.item()
                    num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        print(f"[Client {self.client_id}] local training done. avg_loss={avg_loss:.4f}")

        return self.get_parameters(config={}), len(self.examples), {"loss": avg_loss}

    def evaluate(self, parameters, config):
        """Minimal evaluation — reuses training loss as a rough signal.
        Good enough for a demo-scale project; not a held-out test set."""
        self.set_parameters(parameters)
        self.model.eval()

        total_loss = 0.0
        num_steps = 0
        with torch.no_grad():
            for example in self.examples[:10]:  # quick check on a small subset
                tokens = format_example(example, self.tokenizer)
                input_ids = tokens["input_ids"].to("cuda")
                attention_mask = tokens["attention_mask"].to("cuda")
                outputs = self.model(
                    input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
                )
                total_loss += outputs.loss.item()
                num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        return avg_loss, len(self.examples), {"loss": avg_loss}


def main():
    # client_id is passed as a command-line argument: python client.py 0
    client_id = int(sys.argv[1]) if len(sys.argv) > 1 else 0
    client = FedFlowClient(client_id)
    fl.client.start_client(
        server_address="127.0.0.1:8080",
        client=client.to_client(),
    )


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'flwr'

In [ ]:
!pip install torch transformers peft accelerate bitsandbytes flwr[simulation] -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/2

In [ ]:
!pip uninstall -y torch -q
!pip install torch --index-url https://download.pytorch.org/whl/cu121 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 224.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 241.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 195.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 73.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 70.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 74.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 111.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 110.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 110.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip uninstall -y torchvision torchaudio -q
!pip install "transformers==4.46.3" "peft==0.13.2" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 92.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.29 requires torchvision, which is not installed.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
!python3 -c "import torch; print('CUDA:', torch.cuda.is_available()); print('torch:', torch.__version__)"

CUDA: True
torch: 2.5.1+cu121


In [ ]:
!python3 -m py_compile client.py server.py && echo "Both files are syntactically valid."

[Errno 2] No such file or directory: 'client.py'

In [ ]:
"""
FedFlow-LLM — client.py
------------------------
This defines what ONE simulated client (site) does:
  1. Load the shared base model + a small LoRA adapter on top of it.
  2. When the server sends the current adapter weights, load them in.
  3. Train locally, for a few steps, on this client's own slice of data.
  4. Send the updated adapter weights back to the server.

Nothing here is Flower-specific magic — it's a normal PyTorch/HuggingFace
training loop, wrapped in the shape Flower expects (a "NumPyClient").
"""

# Must be set BEFORE importing transformers/peft — these environment
# variables tell transformers to skip its optional vision/audio code
# paths entirely, rather than trying to import torchvision/torchaudio.
# This project is text-only (traffic forecasting), so none of this is
# needed — and unlike patching torchvision/torchaudio one at a time,
# this blocks the whole multimodal import path in one shot.
import os
os.environ["USE_TORCHVISION"] = "0"
os.environ["USE_TORCHAUDIO"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import json
import sys

import flwr as fl
import numpy as np
import torch
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, get_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # swapped from 0.5B — matches the Objectives slide's stated target
DATA_PATH = "data/traffic_train.json"
NUM_CLIENTS = 6
LOCAL_EPOCHS = 1          # how many passes over this client's own data slice, per round
LOCAL_BATCH_SIZE = 2
MAX_LENGTH = 256           # max tokens per training example (keeps this fast)


def load_model_and_tokenizer():
    """Load the base model once, and wrap it with a small LoRA adapter.
    Only the LoRA adapter's weights get trained/shared — the base model
    itself stays frozen, which is what keeps this small and fast."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cuda",
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],  # standard, small, fast LoRA target
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    return model, tokenizer


def load_client_data(client_id: int, num_clients: int):
    """Split the shared training JSON into num_clients roughly-equal
    slices, and return only this client's slice — simulating each
    client only ever seeing its own local data."""
    with open(DATA_PATH, "r") as f:
        all_examples = json.load(f)

    # Deterministic split: client_id 0 gets every 0th, num_clients-th example, etc.
    my_examples = all_examples[client_id::num_clients]
    return my_examples


def format_example(example, tokenizer):
    """Turn one {"instruction","input","output"} example into a single
    tokenized training string: instruction followed by the expected output."""
    full_text = example["instruction"] + "\n" + example["output"] + tokenizer.eos_token
    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    return tokens


class FedFlowClient(fl.client.NumPyClient):
    def __init__(self, client_id: int):
        self.client_id = client_id
        self.model, self.tokenizer = load_model_and_tokenizer()
        self.examples = load_client_data(client_id, NUM_CLIENTS)
        print(f"[Client {client_id}] loaded {len(self.examples)} local examples.")

    def get_parameters(self, config):
        """Send the current LoRA adapter weights to the server, as a list of numpy arrays."""
        state_dict = get_peft_model_state_dict(self.model)
        return [val.cpu().numpy() for val in state_dict.values()]

    def set_parameters(self, parameters):
        """Receive the server's current (averaged) LoRA weights and load them into this client's model."""
        state_dict = get_peft_model_state_dict(self.model)
        new_state_dict = {
            key: torch.tensor(val) for key, val in zip(state_dict.keys(), parameters)
        }
        set_peft_model_state_dict(self.model, new_state_dict)

    def fit(self, parameters, config):
        """This is the core federated step:
        1. Load the server's current weights.
        2. Train locally on this client's own data slice.
        3. Return the updated weights, plus how many examples were trained on
           (Flower uses this count to weight the averaging — FedAvg)."""
        self.set_parameters(parameters)
        self.model.train()

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4)

        total_loss = 0.0
        num_steps = 0

        for epoch in range(LOCAL_EPOCHS):
            for i in range(0, len(self.examples), LOCAL_BATCH_SIZE):
                batch = self.examples[i : i + LOCAL_BATCH_SIZE]
                if not batch:
                    continue

                for example in batch:
                    tokens = format_example(example, self.tokenizer)
                    input_ids = tokens["input_ids"].to("cuda")
                    attention_mask = tokens["attention_mask"].to("cuda")

                    outputs = self.model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=input_ids,
                    )
                    loss = outputs.loss

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    total_loss += loss.item()
                    num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        print(f"[Client {self.client_id}] local training done. avg_loss={avg_loss:.4f}")

        return self.get_parameters(config={}), len(self.examples), {"loss": avg_loss}

    def evaluate(self, parameters, config):
        """Minimal evaluation — reuses training loss as a rough signal.
        Good enough for a demo-scale project; not a held-out test set."""
        self.set_parameters(parameters)
        self.model.eval()

        total_loss = 0.0
        num_steps = 0
        with torch.no_grad():
            for example in self.examples[:10]:  # quick check on a small subset
                tokens = format_example(example, self.tokenizer)
                input_ids = tokens["input_ids"].to("cuda")
                attention_mask = tokens["attention_mask"].to("cuda")
                outputs = self.model(
                    input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
                )
                total_loss += outputs.loss.item()
                num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        return avg_loss, len(self.examples), {"loss": avg_loss}


def main():
    # client_id is passed as a command-line argument: python client.py 0
    client_id = int(sys.argv[1]) if len(sys.argv) > 1 else 0
    client = FedFlowClient(client_id)
    fl.client.start_client(
        server_address="127.0.0.1:8080",
        client=client.to_client(),
    )


if __name__ == "__main__":
    main()

ValueError: invalid literal for int() with base 10: '-f'

In [ ]:

%%writefile client.py

Writing client.py


In [ ]:
%%writefile client.py

"""
FedFlow-LLM — client.py
------------------------
This defines what ONE simulated client (site) does:
  1. Load the shared base model + a small LoRA adapter on top of it.
  2. When the server sends the current adapter weights, load them in.
  3. Train locally, for a few steps, on this client's own slice of data.
  4. Send the updated adapter weights back to the server.

Nothing here is Flower-specific magic — it's a normal PyTorch/HuggingFace
training loop, wrapped in the shape Flower expects (a "NumPyClient").
"""

# Must be set BEFORE importing transformers/peft — these environment
# variables tell transformers to skip its optional vision/audio code
# paths entirely, rather than trying to import torchvision/torchaudio.
# This project is text-only (traffic forecasting), so none of this is
# needed — and unlike patching torchvision/torchaudio one at a time,
# this blocks the whole multimodal import path in one shot.
import os
os.environ["USE_TORCHVISION"] = "0"
os.environ["USE_TORCHAUDIO"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import json
import sys

import flwr as fl
import numpy as np
import torch
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, get_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # swapped from 0.5B — matches the Objectives slide's stated target
DATA_PATH = "data/traffic_train.json"
NUM_CLIENTS = 6
LOCAL_EPOCHS = 1          # how many passes over this client's own data slice, per round
LOCAL_BATCH_SIZE = 2
MAX_LENGTH = 256           # max tokens per training example (keeps this fast)


def load_model_and_tokenizer():
    """Load the base model once, and wrap it with a small LoRA adapter.
    Only the LoRA adapter's weights get trained/shared — the base model
    itself stays frozen, which is what keeps this small and fast."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cuda",
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],  # standard, small, fast LoRA target
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    return model, tokenizer


def load_client_data(client_id: int, num_clients: int):
    """Split the shared training JSON into num_clients roughly-equal
    slices, and return only this client's slice — simulating each
    client only ever seeing its own local data."""
    with open(DATA_PATH, "r") as f:
        all_examples = json.load(f)

    # Deterministic split: client_id 0 gets every 0th, num_clients-th example, etc.
    my_examples = all_examples[client_id::num_clients]
    return my_examples


def format_example(example, tokenizer):
    """Turn one {"instruction","input","output"} example into a single
    tokenized training string: instruction followed by the expected output."""
    full_text = example["instruction"] + "\n" + example["output"] + tokenizer.eos_token
    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    return tokens


class FedFlowClient(fl.client.NumPyClient):
    def __init__(self, client_id: int):
        self.client_id = client_id
        self.model, self.tokenizer = load_model_and_tokenizer()
        self.examples = load_client_data(client_id, NUM_CLIENTS)
        print(f"[Client {client_id}] loaded {len(self.examples)} local examples.")

    def get_parameters(self, config):
        """Send the current LoRA adapter weights to the server, as a list of numpy arrays."""
        state_dict = get_peft_model_state_dict(self.model)
        return [val.cpu().numpy() for val in state_dict.values()]

    def set_parameters(self, parameters):
        """Receive the server's current (averaged) LoRA weights and load them into this client's model."""
        state_dict = get_peft_model_state_dict(self.model)
        new_state_dict = {
            key: torch.tensor(val) for key, val in zip(state_dict.keys(), parameters)
        }
        set_peft_model_state_dict(self.model, new_state_dict)

    def fit(self, parameters, config):
        """This is the core federated step:
        1. Load the server's current weights.
        2. Train locally on this client's own data slice.
        3. Return the updated weights, plus how many examples were trained on
           (Flower uses this count to weight the averaging — FedAvg)."""
        self.set_parameters(parameters)
        self.model.train()

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4)

        total_loss = 0.0
        num_steps = 0

        for epoch in range(LOCAL_EPOCHS):
            for i in range(0, len(self.examples), LOCAL_BATCH_SIZE):
                batch = self.examples[i : i + LOCAL_BATCH_SIZE]
                if not batch:
                    continue

                for example in batch:
                    tokens = format_example(example, self.tokenizer)
                    input_ids = tokens["input_ids"].to("cuda")
                    attention_mask = tokens["attention_mask"].to("cuda")

                    outputs = self.model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=input_ids,
                    )
                    loss = outputs.loss

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    total_loss += loss.item()
                    num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        print(f"[Client {self.client_id}] local training done. avg_loss={avg_loss:.4f}")

        return self.get_parameters(config={}), len(self.examples), {"loss": avg_loss}

    def evaluate(self, parameters, config):
        """Minimal evaluation — reuses training loss as a rough signal.
        Good enough for a demo-scale project; not a held-out test set."""
        self.set_parameters(parameters)
        self.model.eval()

        total_loss = 0.0
        num_steps = 0
        with torch.no_grad():
            for example in self.examples[:10]:  # quick check on a small subset
                tokens = format_example(example, self.tokenizer)
                input_ids = tokens["input_ids"].to("cuda")
                attention_mask = tokens["attention_mask"].to("cuda")
                outputs = self.model(
                    input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
                )
                total_loss += outputs.loss.item()
                num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        return avg_loss, len(self.examples), {"loss": avg_loss}


def main():
    # client_id is passed as a command-line argument: python client.py 0
    client_id = int(sys.argv[1]) if len(sys.argv) > 1 else 0
    client = FedFlowClient(client_id)
    fl.client.start_client(
        server_address="127.0.0.1:8080",
        client=client.to_client(),
    )


if __name__ == "__main__":
    main()

Overwriting client.py


In [ ]:
!python3 -m py_compile client.py server.py && echo "Both files are syntactically valid."

[Errno 2] No such file or directory: 'server.py'

In [ ]:
%%writefile server.py

"""
FedFlow-LLM — server.py
------------------------
This runs the federated SIMULATION: it starts all 6 clients as
simulated processes on this one Colab GPU, coordinates NUM_ROUNDS
rounds of: send weights out -> clients train locally -> collect
updates back -> average them (FedAvg) -> repeat.

You don't call client.py directly — this file spins up all the
clients itself, in simulation mode, which is the standard way to
run federated learning demos on a single machine.
"""

import flwr as fl
from flwr.simulation import run_simulation
from flwr.client import ClientApp
from flwr.server import ServerApp, ServerConfig
from flwr.server.strategy import FedAvg

from client import FedFlowClient, NUM_CLIENTS

NUM_ROUNDS = 25  # Day 3 first test: keep this small (5). Bump to 20-25 for the real evidence run.


def client_fn(context):
    """Flower calls this once per simulated client to create it.
    context.node_config["partition-id"] tells us which client this is (0 to NUM_CLIENTS-1)."""
    client_id = int(context.node_config["partition-id"])
    return FedFlowClient(client_id).to_client()


def server_fn(context):
    """Defines the FedAvg strategy and how many rounds to run."""
    strategy = FedAvg(
        fraction_fit=1.0,          # use all 6 clients every round
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
    )
    config = ServerConfig(num_rounds=NUM_ROUNDS)
    return fl.server.ServerAppComponents(strategy=strategy, config=config)


client_app = ClientApp(client_fn=client_fn)
server_app = ServerApp(server_fn=server_fn)


if __name__ == "__main__":
    print(f"Starting FedFlow-LLM simulation: {NUM_CLIENTS} clients, {NUM_ROUNDS} rounds.")
    run_simulation(
        server_app=server_app,
        client_app=client_app,
        num_supernodes=NUM_CLIENTS,
        backend_config={"client_resources": {"num_cpus": 1, "num_gpus": 1.0 / NUM_CLIENTS}},
    )

Writing server.py


In [ ]:
!python3 -m py_compile client.py server.py && echo "Both files are syntactically valid."

Both files are syntactically valid.


In [ ]:
!python server.py

Starting FedFlow-LLM simulation: 6 clients, 25 rounds.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
DEBUG:flwr:backend_config: {'client_resources': {'num_cpus': 1, 'num_gpus': 0.16666666666666666}}
DEBUG:flwr:Pre-registering run with id 12996551865023005844
DEBUG:flwr:Using InMemoryObjectStore
DEBUG:flwr:Using InMemoryLinkState
INFO :      Starting Flower ServerApp, config: num_rounds=25, no round_timeout
INFO:flwr:Starting Flower ServerApp, config: num_rounds=25, no round_timeout
INFO :      
INFO:flwr:
INFO :      [INIT]
DEBUG:flwr:Registered 6 nodes
INFO:flwr:[INIT]
INFO :      Requesting initial parameters from one random client
INFO:flwr:Requesting initial parameters from one random client
DEBUG:flwr:Initialising: RayBackend
DEBUG:flwr:Backend config: {'client_resources': {'num_cpus': 

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

# Check the existing scripts without executing or modifying them.
for filename, pattern in [
    ("client.py", r"""(?m)^MODEL_NAME\s*=\s*["']Qwen/Qwen2\.5-1\.5B-Instruct["']"""),
    ("server.py", r"(?m)^NUM_ROUNDS\s*=\s*25\s*(?:#.*)?$"),
]:
    path = Path(filename)
    if not path.exists():
        raise RuntimeError(f"{filename} is missing. Stop here and share this error.")
    if not re.search(pattern, path.read_text()):
        raise RuntimeError(f"{filename} has unexpected settings. Nothing was changed.")

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
csv_path = data_dir / "synthetic_traffic.csv"
json_path = data_dir / "traffic_train.json"

if not json_path.exists():
    if not csv_path.exists():
        # Reproduce the original synthetic dataset with the original seed.
        rng = np.random.default_rng(42)
        n_steps = 14 * 144
        t = np.arange(n_steps)
        base = 100 + 35 * np.sin(2 * np.pi * (t % 144) / 144 - np.pi / 2)
        traffic = base * np.where((t // 144) % 7 >= 5, 0.7, 1.0)
        traffic += rng.normal(0, 7, n_steps)

        for _ in range(15):
            start = rng.integers(0, n_steps - 8)
            duration = rng.integers(2, 9)
            height = rng.uniform(30, 120)
            traffic[start:start + duration] += (
                height * np.hanning(duration + 2)[1:-1]
            )

        pd.DataFrame({
            "timestep": t,
            "traffic": np.clip(traffic, 0, None).round(2),
        }).to_csv(csv_path, index=False)
        print("Restored:", csv_path)

    df = pd.read_csv(csv_path)
    if len(df) < 217 or not np.isfinite(df["traffic"].to_numpy(dtype=float)).all():
        raise ValueError("CSV must contain at least 217 rows of finite traffic values.")

    rng = np.random.default_rng(123)
    examples, expected = [], []

    for i in rng.choice(len(df) - 17, size=200, replace=False):
        history = df["traffic"].iloc[i:i + 12].tolist()
        future = df["traffic"].iloc[i + 12:i + 18].tolist()
        slope = float(np.mean(history[-3:]) - np.mean(history[:3]))
        trend = "increasing" if slope > 1 else "decreasing" if slope < -1 else "stable"

        instruction = (
            "You are a traffic-forecasting assistant. Predict the next 6 traffic values "
            "given the following 12 historical traffic values measured at 10-minute intervals.\n\n"
            f"History: {', '.join(f'{v:.2f}' for v in history)}\n\n"
            "Reason step by step about the traffic pattern, then end with exactly this format:\n"
            "PREDICTION: v1, v2, v3, v4, v5, v6"
        )
        output = (
            f"Trend direction: {trend}.\n"
            f"Justification: The history slope (mean of last 3 minus mean of first 3) is {slope:.2f}.\n"
            f"PREDICTION: {', '.join(f'{v:.2f}' for v in future)}"
        )
        examples.append({"instruction": instruction, "input": "", "output": output})
        expected.append([float(f"{v:.2f}") for v in future])

    for index, (example, target) in enumerate(zip(examples, expected)):
        parsed = [
            float(v) for v in example["output"].split("PREDICTION:", 1)[1].split(",")
        ]
        if len(parsed) != 6 or parsed != target:
            raise AssertionError(f"Verification failed at example {index}.")

    with json_path.open("x", encoding="utf-8") as f:
        json.dump(examples, f, indent=2)
    print("Restored and verified 200 examples:", json_path)
else:
    print("Keeping existing dataset:", json_path)

with json_path.open(encoding="utf-8") as f:
    examples = json.load(f)

if not isinstance(examples, list) or len(examples) != 200:
    raise ValueError("Expected a JSON list containing 200 examples.")
if not all(
    isinstance(e, dict)
    and all(isinstance(e.get(k), str) for k in ("instruction", "input", "output"))
    for e in examples
):
    raise ValueError("Dataset contains malformed examples.")

print("Ready: Qwen 1.5B | 6-client simulation configured in server.py | 25 rounds | 200 examples.")

Restored: data/synthetic_traffic.csv
Restored and verified 200 examples: data/traffic_train.json
Ready: Qwen 1.5B | 6-client simulation configured in server.py | 25 rounds | 200 examples.


In [ ]:
!python server.py

Starting FedFlow-LLM simulation: 6 clients, 25 rounds.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
DEBUG:flwr:backend_config: {'client_resources': {'num_cpus': 1, 'num_gpus': 0.16666666666666666}}
DEBUG:flwr:Pre-registering run with id 8919109340382012046
DEBUG:flwr:Using InMemoryObjectStore
DEBUG:flwr:Using InMemoryLinkState
INFO :      Starting Flower ServerApp, config: num_rounds=25, no round_timeout
INFO:flwr:Starting Flower ServerApp, config: num_rounds=25, no round_timeout
INFO :      
INFO:flwr:
DEBUG:flwr:Registered 6 nodes
INFO :      [INIT]
INFO:flwr:[INIT]
INFO :      Requesting initial parameters from one random client
INFO:flwr:Requesting initial parameters from one random client
DEBUG:flwr:Initialising: RayBackend
DEBUG:flwr:Backend config: {'client_resources': {'num_cpus': 1

In [ ]:
!nvidia-smi


Sun Sep 27 17:12:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install torch transformers peft accelerate bitsandbytes flwr[simulation] -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/2

In [ ]:
!pip uninstall -y torch -q
!pip install torch --index-url https://download.pytorch.org/whl/cu121 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 213.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 235.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 238.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 78.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 135.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 76.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 108.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 70.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 88.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip uninstall -y torchvision torchaudio -q
!pip install "transformers==4.46.3" "peft==0.13.2" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.29 requires torchvision, which is not installed.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
!python3 -c "import torch; print('CUDA:', torch.cuda.is_available()); print('torch:', torch.__version__)"

CUDA: True
torch: 2.5.1+cu121


In [ ]:
%%writefile client.py

"""
FedFlow-LLM — client.py
------------------------
This defines what ONE simulated client (site) does:
  1. Load the shared base model + a small LoRA adapter on top of it.
  2. When the server sends the current adapter weights, load them in.
  3. Train locally, for a few steps, on this client's own slice of data.
  4. Send the updated adapter weights back to the server.

Nothing here is Flower-specific magic — it's a normal PyTorch/HuggingFace
training loop, wrapped in the shape Flower expects (a "NumPyClient").
"""

# Must be set BEFORE importing transformers/peft — these environment
# variables tell transformers to skip its optional vision/audio code
# paths entirely, rather than trying to import torchvision/torchaudio.
# This project is text-only (traffic forecasting), so none of this is
# needed — and unlike patching torchvision/torchaudio one at a time,
# this blocks the whole multimodal import path in one shot.
import os
os.environ["USE_TORCHVISION"] = "0"
os.environ["USE_TORCHAUDIO"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import json
import sys

import flwr as fl
import numpy as np
import torch
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, get_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # matches the Objectives slide's stated target
DATA_PATH = "data/traffic_train.json"
NUM_CLIENTS = 6
LOCAL_EPOCHS = 1          # how many passes over this client's own data slice, per round
LOCAL_BATCH_SIZE = 2
MAX_LENGTH = 256           # max tokens per training example (keeps this fast)


def load_model_and_tokenizer():
    """Load the base model once, and wrap it with a small LoRA adapter.
    Only the LoRA adapter's weights get trained/shared — the base model
    itself stays frozen, which is what keeps this small and fast."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cuda",
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],  # standard, small, fast LoRA target
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    return model, tokenizer


def load_client_data(client_id: int, num_clients: int):
    """Split the shared training JSON into num_clients roughly-equal
    slices, and return only this client's slice — simulating each
    client only ever seeing its own local data."""
    with open(DATA_PATH, "r") as f:
        all_examples = json.load(f)

    # Deterministic split: client_id 0 gets every 0th, num_clients-th example, etc.
    my_examples = all_examples[client_id::num_clients]
    return my_examples


def format_example(example, tokenizer):
    """Turn one {"instruction","input","output"} example into a single
    tokenized training string: instruction followed by the expected output."""
    full_text = example["instruction"] + "\n" + example["output"] + tokenizer.eos_token
    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    return tokens


class FedFlowClient(fl.client.NumPyClient):
    def __init__(self, client_id: int):
        self.client_id = client_id
        self.model, self.tokenizer = load_model_and_tokenizer()
        self.examples = load_client_data(client_id, NUM_CLIENTS)
        print(f"[Client {client_id}] loaded {len(self.examples)} local examples.")

    def get_parameters(self, config):
        """Send the current LoRA adapter weights to the server, as a list of numpy arrays."""
        state_dict = get_peft_model_state_dict(self.model)
        return [val.cpu().numpy() for val in state_dict.values()]

    def set_parameters(self, parameters):
        """Receive the server's current (averaged) LoRA weights and load them into this client's model."""
        state_dict = get_peft_model_state_dict(self.model)
        new_state_dict = {
            key: torch.tensor(val) for key, val in zip(state_dict.keys(), parameters)
        }
        set_peft_model_state_dict(self.model, new_state_dict)

    def fit(self, parameters, config):
        """This is the core federated step:
        1. Load the server's current weights.
        2. Train locally on this client's own data slice.
        3. Return the updated weights, plus how many examples were trained on
           (Flower uses this count to weight the averaging — FedAvg)."""
        self.set_parameters(parameters)
        self.model.train()

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4)

        total_loss = 0.0
        num_steps = 0

        for epoch in range(LOCAL_EPOCHS):
            for i in range(0, len(self.examples), LOCAL_BATCH_SIZE):
                batch = self.examples[i : i + LOCAL_BATCH_SIZE]
                if not batch:
                    continue

                for example in batch:
                    tokens = format_example(example, self.tokenizer)
                    input_ids = tokens["input_ids"].to("cuda")
                    attention_mask = tokens["attention_mask"].to("cuda")

                    outputs = self.model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=input_ids,
                    )
                    loss = outputs.loss

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    total_loss += loss.item()
                    num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        print(f"[Client {self.client_id}] local training done. avg_loss={avg_loss:.4f}")

        return self.get_parameters(config={}), len(self.examples), {"loss": avg_loss}

    def evaluate(self, parameters, config):
        """Minimal evaluation — reuses training loss as a rough signal.
        Good enough for a demo-scale project; not a held-out test set."""
        self.set_parameters(parameters)
        self.model.eval()

        total_loss = 0.0
        num_steps = 0
        with torch.no_grad():
            for example in self.examples[:10]:  # quick check on a small subset
                tokens = format_example(example, self.tokenizer)
                input_ids = tokens["input_ids"].to("cuda")
                attention_mask = tokens["attention_mask"].to("cuda")
                outputs = self.model(
                    input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
                )
                total_loss += outputs.loss.item()
                num_steps += 1

        avg_loss = total_loss / max(num_steps, 1)
        return avg_loss, len(self.examples), {"loss": avg_loss}


def main():
    # client_id is passed as a command-line argument: python client.py 0
    client_id = int(sys.argv[1]) if len(sys.argv) > 1 else 0
    client = FedFlowClient(client_id)
    fl.client.start_client(
        server_address="127.0.0.1:8080",
        client=client.to_client(),
    )


if __name__ == "__main__":
    main()

Writing client.py


In [ ]:
%%writefile server.py

"""
FedFlow-LLM — server.py
------------------------
This runs the federated SIMULATION: it starts all 6 clients as
simulated processes on this one Colab GPU, coordinates NUM_ROUNDS
rounds of: send weights out -> clients train locally -> collect
updates back -> average them (FedAvg) -> repeat.

You don't call client.py directly — this file spins up all the
clients itself, in simulation mode, which is the standard way to
run federated learning demos on a single machine.
"""

import flwr as fl
from flwr.simulation import run_simulation
from flwr.client import ClientApp
from flwr.server import ServerApp, ServerConfig
from flwr.server.strategy import FedAvg

from client import FedFlowClient, NUM_CLIENTS

NUM_ROUNDS = 25  # Real evidence run — both 0.5B and 1.5B confirmed working at this round count.


def client_fn(context):
    """Flower calls this once per simulated client to create it.
    context.node_config["partition-id"] tells us which client this is (0 to NUM_CLIENTS-1)."""
    client_id = int(context.node_config["partition-id"])
    return FedFlowClient(client_id).to_client()


def server_fn(context):
    """Defines the FedAvg strategy and how many rounds to run."""
    strategy = FedAvg(
        fraction_fit=1.0,          # use all 6 clients every round
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
    )
    config = ServerConfig(num_rounds=NUM_ROUNDS)
    return fl.server.ServerAppComponents(strategy=strategy, config=config)


client_app = ClientApp(client_fn=client_fn)
server_app = ServerApp(server_fn=server_fn)


if __name__ == "__main__":
    print(f"Starting FedFlow-LLM simulation: {NUM_CLIENTS} clients, {NUM_ROUNDS} rounds.")
    run_simulation(
        server_app=server_app,
        client_app=client_app,
        num_supernodes=NUM_CLIENTS,
        backend_config={"client_resources": {"num_cpus": 1, "num_gpus": 1.0 / NUM_CLIENTS}},
    )

Writing server.py


In [ ]:
!python3 -m py_compile client.py server.py && echo "Both files are syntactically valid."

Both files are syntactically valid.


In [ ]:
!nvidia-smi

Sun Sep 27 18:18:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
print(dir())

['In', 'Out', '_', '__', '___', '__builtin__', '__builtins__', '__doc__', '__loader__', '__name__', '__package__', '__spec__', '_dh', '_exit_code', '_i', '_i1', '_i2', '_ih', '_ii', '_iii', '_oh', 'exit', 'get_ipython', 'quit']


In [ ]:
import os
print(os.path.exists('/content/drive'))

False


In [ ]:
!pip install -q torch transformers peft accelerate bitsandbytes "flwr[simulation]"
!pip uninstall -y torchvision torchaudio -q
!pip install -q transformers==4.46.3 peft==0.13.2

In [ ]:
!pip install -q torch transformers peft accelerate bitsandbytes "flwr[simulation]"
!pip uninstall -y torchvision torchaudio -q
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.46.3 peft==0.13.2

In [ ]:
import numpy as np
import pandas as pd
import json
import os

np.random.seed(42)
n_points = 14 * 24 * 6
t = np.arange(n_points)
diurnal = 50 + 40 * np.sin(2 * np.pi * t / 144 - np.pi/2)
diurnal = np.clip(diurnal, 5, None)
day_idx = (t // 144) % 7
weekend_mask = np.isin(day_idx, [5, 6])
diurnal[weekend_mask] *= 0.7
noise = np.random.normal(0, 5, n_points)
traffic = diurnal + noise
n_bursts = 15
burst_starts = np.random.choice(n_points - 20, n_bursts, replace=False)
for b in burst_starts:
    burst_len = np.random.randint(3, 12)
    burst_height = np.random.uniform(30, 80)
    traffic[b:b+burst_len] += burst_height
traffic = np.clip(traffic, 0, None)

df = pd.DataFrame({"timestep": t, "traffic": traffic.round(2)})
os.makedirs("data", exist_ok=True)
df.to_csv("data/synthetic_traffic.csv", index=False)
print(f"Generated {len(df)} points. Mean: {df.traffic.mean():.1f}, Max: {df.traffic.max():.1f}")

Generated 2016 points. Mean: 48.3, Max: 167.7
